# Agent with Memory Tools

## The Combined Pattern: Passive + Active Memory

The previous notebook used a context provider for **passive** memory injection — it runs automatically every turn, silently providing context. This notebook adds **active** memory tools that the agent explicitly decides to call.

The combined pattern gives the agent both:
- **Automatic context** (context provider) — relevant history and entities injected before every turn
- **Explicit control** (tools) — the agent can actively search, save, and recall information

This notebook uses the [`neo4j-agent-memory`](https://github.com/neo4j-labs/agent-memory) package with `create_memory_tools()` to generate callable `FunctionTool` instances:

## What you will learn

- How to create memory tools with `create_memory_tools()`
- How to combine context providers with tools in a single agent
- How the agent decides when to call memory tools vs. relying on injected context
- The available memory tools: `search_memory`, `remember_preference`, `recall_preferences`, `search_knowledge`, `remember_fact`, `find_similar_tasks`

***

Load the environment variables and import the required modules.

In [ ]:
import sys
sys.path.insert(0, '../shared')

import asyncio

from pydantic import SecretStr

from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

from neo4j_agent_memory import MemoryClient, MemorySettings
from neo4j_agent_memory.integrations.microsoft_agent import (
    Neo4jMicrosoftMemory,
    create_memory_tools,
)

from azure_embedder import get_memory_embedder
from config import get_agent_config, Neo4jConfig

## Set Up Memory with Tools

Create the memory client and unified memory interface, then generate callable memory tools.

In [ ]:
# Get workshop configuration
config = get_agent_config()
neo4j_config = Neo4jConfig()

# Configure and create memory client
settings = MemorySettings(
    neo4j={
        "uri": neo4j_config.uri,
        "username": neo4j_config.username,
        "password": SecretStr(neo4j_config.password),
    },
    extraction={
        "enable_gliner": False,  # GLiNER disabled: downloads ~500MB model, impractical in a workshop
    },
)

memory_client = MemoryClient(settings, embedder=get_memory_embedder())
await memory_client.__aenter__()

# Create unified memory
memory = Neo4jMicrosoftMemory.from_memory_client(
    memory_client=memory_client,
    session_id="workshop-tools-demo",
    include_short_term=True,
    include_long_term=True,
    include_reasoning=True,
    extract_entities=True,
)

# Create callable memory tools
tools = create_memory_tools(memory)

print(f"Created {len(tools)} memory tools:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

> These tools are `FunctionTool` instances that the Microsoft Agent Framework auto-invokes during streaming. The agent decides when to use each tool based on the tool's name and description.

***

## Create the Agent with Memory Tools and Context Provider

Combine the context provider (automatic background context) with memory tools (explicit agent-driven operations).

In [ ]:
async def run_memory_agent():
    async with AzureCliCredential() as credential:
        async with AzureAIClient(
            project_endpoint=config.project_endpoint,
            model_deployment_name=config.model_name,
            credential=credential,
        ) as client:
            agent = client.as_agent(
                name="workshop-memory-tools-agent",
                instructions=(
                    "You are a helpful assistant with persistent memory. You have access to "
                    "memory tools that let you:\n"
                    "1. Search your memory for relevant past conversations and facts\n"
                    "2. Save user preferences when they express them\n"
                    "3. Recall preferences to personalize your responses\n"
                    "4. Search the knowledge graph for entities\n"
                    "5. Remember important facts for future reference\n\n"
                    "IMPORTANT: You MUST call the remember_preference tool every time "
                    "a user states a preference, favorite, or interest. Do NOT just "
                    "acknowledge it verbally — you must actually invoke the tool to "
                    "persist it. For example, if a user says they prefer concise "
                    "explanations, call remember_preference with "
                    "category='communication' and preference='prefers concise "
                    "technical explanations'.\n\n"
                    "When making recommendations or answering questions, use "
                    "recall_preferences to check what the user likes before responding."
                ),
                tools=tools,
                context_providers=[memory.context_provider],
            )
            session = agent.create_session()

            queries = [
                "I prefer concise technical explanations over high-level overviews.",
                "What can you tell me about supply chain risks in tech companies?",
                "Remember that I'm particularly interested in Apple and Microsoft.",
                "Based on what you know about my preferences, what should I focus on?",
            ]

            for query in queries:
                print(f"User: {query}\n")
                print("Assistant: ", end="", flush=True)

                async for update in agent.run(query, stream=True, session=session):
                    if update.text:
                        print(update.text, end="", flush=True)

                print("\n\n" + "-" * 50 + "\n")

    await asyncio.sleep(0.1)

await run_memory_agent()

> The agent used memory tools to save preferences and recall them later. The context provider also injected relevant conversation history automatically.

***

## Verify Stored Memories

Let's check what the agent stored in memory.

In [ ]:
# Search for stored preferences
results = await memory.search_memory(
    query="user preferences and interests",
    include_messages=True,
    include_entities=True,
    include_preferences=True,
    limit=5,
)

print("=== Stored Memories ===\n")

if results.get("preferences"):
    print("Preferences:")
    for pref in results["preferences"]:
        print(f"  [{pref['category']}] {pref['preference']}")
    print()

if results.get("entities"):
    print("Entities:")
    for entity in results["entities"][:5]:
        print(f"  {entity['name']} ({entity['type']})")
    print()

if results.get("messages"):
    print(f"Messages stored: {len(results['messages'])}")

## Experiment

- Express different preferences and check if the agent saves them using `remember_preference`
- Ask the agent "What do you know about me?" to test `recall_preferences`
- Try a multi-turn conversation where the agent uses recalled preferences to tailor its responses

***

[View the complete code](../financial_data_load/solution_srcs/07_03_memory_tools_agent.py)

[Continue to the Reasoning Memory Notebook](04_reasoning_memory.ipynb)

In [ ]:
# Cleanup
await memory_client.__aexit__(None, None, None)